**Initialization and Helper Functions**

In [1]:
import math

# Target tolerance for 7 significant digits
TOL = 5e-8

# Helper function to check stopping criteria
def check_convergence(curr, prev, f_curr, criterion, tol=TOL):
    if criterion == 'Absolute Error':
        return abs(curr - prev) < tol
    elif criterion == 'Relative Error':
        return abs(curr - prev) / abs(curr) < tol if curr != 0 else False
    elif criterion == 'Functional value':
        return abs(f_curr) < tol
    return False

# Function for Q2, Q3, and Q4: f(x) = cos(x) - x*e^x
def f(x):
    return math.cos(x) - x * math.exp(x)

# Derivative for Newton-Raphson (Q3): f'(x) = -sin(x) - e^x * (x + 1)
def df(x):
    return -math.sin(x) - math.exp(x) * (x + 1)

# Function g(x) for Fixed Point Iteration (Q3): x = cos(x) / e^x
def g(x):
    return math.cos(x) * math.exp(-x)

**Question 1 - Bisection Method for $\sqrt{2}$**

Perform the Bisection method upto 50 iteration and find the value of $\sqrt{2}$.

In [2]:
def q1_bisection_sqrt2():
    a, b = 1.0, 2.0

    # Print Table Header
    print(f"{'Iteration':<10} | {'a':<12} | {'b':<12} | {'c=(a+b)/2':<12} | {'f(a)':<12} | {'f(b)':<12} | {'f(c)':<12}")
    print("-" * 92)

    for i in range(51):  # Iterations 0 to 50
        c = (a + b) / 2.0
        fa = a**2 - 2
        fb = b**2 - 2
        fc = c**2 - 2

        # Print Current Iteration Row
        print(f"{i:<10} | {a:<12.9f} | {b:<12.9f} | {c:<12.9f} | {fa:<12.9f} | {fb:<12.9f} | {fc:<12.9f}")

        # Bisection update
        if fa * fc < 0:
            b = c
        else:
            a = c

# Execute Question 1
q1_bisection_sqrt2()

Iteration  | a            | b            | c=(a+b)/2    | f(a)         | f(b)         | f(c)        
--------------------------------------------------------------------------------------------
0          | 1.000000000  | 2.000000000  | 1.500000000  | -1.000000000 | 2.000000000  | 0.250000000 
1          | 1.000000000  | 1.500000000  | 1.250000000  | -1.000000000 | 0.250000000  | -0.437500000
2          | 1.250000000  | 1.500000000  | 1.375000000  | -0.437500000 | 0.250000000  | -0.109375000
3          | 1.375000000  | 1.500000000  | 1.437500000  | -0.109375000 | 0.250000000  | 0.066406250 
4          | 1.375000000  | 1.437500000  | 1.406250000  | -0.109375000 | 0.066406250  | -0.022460938
5          | 1.406250000  | 1.437500000  | 1.421875000  | -0.022460938 | 0.066406250  | 0.021728516 
6          | 1.406250000  | 1.421875000  | 1.414062500  | -0.022460938 | 0.021728516  | -0.000427246
7          | 1.414062500  | 1.421875000  | 1.417968750  | -0.000427246 | 0.021728516  | 0.010635376

**Question 2 - Bisection Method for $cos(x) = x e^x$**

Use the bisection method to find the root of $cos~x=xe^{x}$
, accurate to at least 7
significant digits. (Take the problem of your choice.) Use the following stopping
criterion.
(a) Absolute error
(b) Relative error
(c) Functional value

In [3]:
def bisection_q2(a_init, b_init, criterion):
    a, b = a_init, b_init
    c_prev = a
    iters = 0

    while True:
        c = (a + b) / 2.0
        fc = f(c)

        if check_convergence(c, c_prev, fc, criterion) and iters > 0:
            break

        if f(a) * fc < 0:
            b = c
        else:
            a = c

        c_prev = c
        iters += 1

        # Failsafe for runaway loops
        if iters > 1000:
            break

    return c, iters

def q2_run():
    a, b = 0.0, 1.0
    criteria = ['Absolute Error', 'Relative Error', 'Functional value']

    print(f"{'Stopping Criterion':<20} | {'a':<5} | {'b':<5} | {'Approximate value of root':<26} | {'No of Iteration required':<25}")
    print("-" * 95)

    for crit in criteria:
        root, iterations = bisection_q2(a, b, crit)
        print(f"{crit:<20} | {a:<5.1f} | {b:<5.1f} | {root:<26.8f} | {iterations:<25}")

# Execute Question 2
q2_run()

Stopping Criterion   | a     | b     | Approximate value of root  | No of Iteration required 
-----------------------------------------------------------------------------------------------
Absolute Error       | 0.0   | 1.0   | 0.51775739                 | 24                       
Relative Error       | 0.0   | 1.0   | 0.51775737                 | 25                       
Functional value     | 0.0   | 1.0   | 0.51775736                 | 23                       


**Question 3 - Newton-Raphson & Fixed Point Iteration**

Take the initial guess $x_{0}$ as a mid point of [a, b] and repeat the question no. 2 for N-R method first and then the fixed point iteration method.

In [4]:
def newton_raphson(x0, criterion):
    x = x0
    iters = 0
    while True:
        fx = f(x)
        if criterion == 'Functional value' and check_convergence(x, None, fx, criterion):
            break

        x_new = x - fx / df(x)
        iters += 1

        if criterion != 'Functional value' and check_convergence(x_new, x, f(x_new), criterion):
            x = x_new
            break

        x = x_new
        if iters > 100: break
    return x, iters

def fixed_point_iteration(x0, criterion):
    x = x0
    iters = 0
    while True:
        x_new = g(x)
        iters += 1

        if check_convergence(x_new, x, f(x_new), criterion):
            x = x_new
            break

        x = x_new
        if iters > 500: break # FPI can be slower
    return x, iters

def q3_run():
    a, b = 0.0, 1.0
    x0 = (a + b) / 2.0
    criteria = ['Absolute Error', 'Relative Error', 'Functional value']

    print("NEWTON-RAPHSON METHOD")
    print(f"{'Criterion':<18} | {'a':<4} | {'b':<4} | {'x0':<4} | {'Approximate value':<18} | {'Iterations'}")
    print("-" * 75)
    for crit in criteria:
        root, iters = newton_raphson(x0, crit)
        print(f"{crit:<18} | {a:<4.1f} | {b:<4.1f} | {x0:<4.1f} | {root:<18.8f} | {iters}")

    print("\nFIXED POINT ITERATION METHOD")
    print(f"{'Criterion':<18} | {'a':<4} | {'b':<4} | {'x0':<4} | {'Approximate value':<18} | {'Iterations'}")
    print("-" * 75)
    for crit in criteria:
        root, iters = fixed_point_iteration(x0, crit)
        print(f"{crit:<18} | {a:<4.1f} | {b:<4.1f} | {x0:<4.1f} | {root:<18.8f} | {iters}")

# Execute Question 3
q3_run()

NEWTON-RAPHSON METHOD
Criterion          | a    | b    | x0   | Approximate value  | Iterations
---------------------------------------------------------------------------
Absolute Error     | 0.0  | 1.0  | 0.5  | 0.51775736         | 4
Relative Error     | 0.0  | 1.0  | 0.5  | 0.51775736         | 4
Functional value   | 0.0  | 1.0  | 0.5  | 0.51775736         | 3

FIXED POINT ITERATION METHOD
Criterion          | a    | b    | x0   | Approximate value  | Iterations
---------------------------------------------------------------------------
Absolute Error     | 0.0  | 1.0  | 0.5  | 0.51775734         | 66
Relative Error     | 0.0  | 1.0  | 0.5  | 0.51775737         | 69
Functional value   | 0.0  | 1.0  | 0.5  | 0.51775738         | 67


**Comments / Observations regarding the methods/results:**


***Convergence Speed:***

The Newton-Raphson method converges significantly faster (requiring only 3–4 iterations) compared to the Bisection method (around 23–25 iterations) and Fixed Point Iteration (around 22–24 iterations). This demonstrates N-R's quadratic convergence rate versus the linear rate of the other two.




***Method Reliability:***

While N-R is the fastest, it requires the calculation of the analytical derivative $f'(x)$. Bisection does not require derivatives and is guaranteed to converge as long as the initial bracket $[a, b]$ satisfies $f(a)f(b) < 0$.







***Fixed Point Iteration Sensitivity:***


The FPI method succeeded here because our choice of $g(x) = \frac{\cos(x)}{e^x}$ satisfied the condition $\vert{}g'(x)\vert{} < 1$ near the root, but it is highly sensitive to how $g(x)$ is constructed.


**Question 4 - Secant and Regula-Falsi Methods**

In assignment-2, repeat the above question 2 with the secant and regula–falsi methods. Finally print the results in tabular forms.

In [5]:
def secant_method(x0, x1, criterion):
    iters = 0
    while True:
        fx0, fx1 = f(x0), f(x1)
        if fx1 - fx0 == 0:
            break

        x2 = x1 - fx1 * (x1 - x0) / (fx1 - fx0)
        iters += 1

        if check_convergence(x2, x1, f(x2), criterion):
            break

        x0, x1 = x1, x2
        if iters > 100: break
    return x2, iters

def regula_falsi(a_init, b_init, criterion):
    a, b = a_init, b_init
    c_prev = a
    iters = 0
    while True:
        fa, fb = f(a), f(b)
        if fb - fa == 0:
            break

        c = b - fb * (b - a) / (fb - fa)
        fc = f(c)
        iters += 1

        if check_convergence(c, c_prev, fc, criterion):
            break

        if fa * fc < 0:
            b = c
        else:
            a = c

        c_prev = c
        if iters > 100: break
    return c, iters

def q4_run():
    a, b = 0.0, 1.0
    criteria = ['Absolute Error', 'Relative Error', 'Functional value']

    print("SECANT METHOD (Using a and b as x0 and x1)")
    print(f"{'Criterion':<18} | {'x0':<4} | {'x1':<4} | {'Approximate value':<18} | {'Iterations'}")
    print("-" * 65)
    for crit in criteria:
        root, iters = secant_method(a, b, crit)
        print(f"{crit:<18} | {a:<4.1f} | {b:<4.1f} | {root:<18.8f} | {iters}")

    print("\nREGULA-FALSI (FALSE POSITION) METHOD")
    print(f"{'Criterion':<18} | {'a':<4} | {'b':<4} | {'Approximate value':<18} | {'Iterations'}")
    print("-" * 65)
    for crit in criteria:
        root, iters = regula_falsi(a, b, crit)
        print(f"{crit:<18} | {a:<4.1f} | {b:<4.1f} | {root:<18.8f} | {iters}")

# Execute Question 4
q4_run()

SECANT METHOD (Using a and b as x0 and x1)
Criterion          | x0   | x1   | Approximate value  | Iterations
-----------------------------------------------------------------
Absolute Error     | 0.0  | 1.0  | 0.51775736         | 7
Relative Error     | 0.0  | 1.0  | 0.51775736         | 7
Functional value   | 0.0  | 1.0  | 0.51775737         | 6

REGULA-FALSI (FALSE POSITION) METHOD
Criterion          | a    | b    | Approximate value  | Iterations
-----------------------------------------------------------------
Absolute Error     | 0.0  | 1.0  | 0.51775735         | 16
Relative Error     | 0.0  | 1.0  | 0.51775735         | 16
Functional value   | 0.0  | 1.0  | 0.51775735         | 16
